In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, \
learning_curve, validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.decomposition import PCA

from scipy.stats import mannwhitneyu

sns.set(font_scale=1.5)
pd.options.display.max_columns = 50

##خطة المشروع
* 1. شرح الميزة والبيانات
* 2. تحليل البيانات الأولية
* 3. تحليل البيانات المرئية الأولية
* 4. الرؤى والتبعيات الموجودة
* 5. اختيار المقاييس
* 6. اختيار النموذج
* 7. المعالجة المسبقة للبيانات
* 8. التحقق من صحة وتعديل المعلمات الفائقة للنموذج
* 9. إنشاء ميزات جديدة ووصف هذه العملية
* 10. رسم منحنيات التدريب والتحقق من الصحة
* 11. التنبؤ بالعينات الاختبارية أو المحتجزة
* 12. الاستنتاجات



## 1. شرح الميزات والبيانات


In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
df.head()


### 1.1 عملية جمع البيانات



يتم حساب الميزات من صورة رقمية لشفطة بإبرة دقيقة (FNA) لكتلة الثدي. يصفون خصائص نواة الخلية الموجودة في الصورة. n الفضاء ثلاثي الأبعاد هو الذي تم وصفه في: [K. P. Bennett and O. L. Mangasarian: "التمييز القوي في البرمجة الخطية لمجموعتين غير قابلتين للفصل خطيًا"، طرق التحسين والبرمجيات 1، 1992، 23-34].
البيانات المستخدمة متاحة من خلال https://www.kaggle.com/uciml/breast-cancer-wisconsin-data  
ويمكن العثور عليها في مستودع التعلم الآلي UCI: https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+%28Diagnostic%29  
قاعدة البيانات هذه متاحة أيضًا من خلال خادم UW CS ftp: ftp ftp.cs.wisc.edu cd math-prog/cpo-dataset/machine-learn/WDBC/



### 1.2 شرح تفصيلي للمهمة



المهمة هنا هي التنبؤ بما إذا كان السرطان حميدًا أم خبيثًا بناءً على 30 سمة ذات قيمة حقيقية.



### 1.3 ميزات الهدف



معلومات السمة:
1) رقم الهوية  
2) التشخيص (M = خبيث، B = حميد)  
3-32)يتم حساب عشر ميزات ذات قيمة حقيقية لكل نواة خلية:  
أ) نصف القطر (متوسط المسافات من المركز إلى النقاط على المحيط)  
ب) الملمس (الانحراف المعياري لقيم التدرج الرمادي)  
ج) محيط  
د) المنطقة  
ه) النعومة (الاختلاف المحلي في أطوال نصف القطر)  
و) الاكتناز (المحيط ^ 2 / المساحة - 1.0)  
ز) التقعر (شدة الأجزاء المقعرة من الكفاف)  
ح) النقاط المقعرة (عدد الأجزاء المقعرة من الكفاف)  
ط) التماثل  
ي) البعد الكسري ("تقريب الخط الساحلي" - 1)  
تم حساب المتوسط والخطأ المعياري و"الأسوأ" أو الأكبر (متوسط القيم الثلاث الكبرى) لهذه الميزات لكل صورة، مما أدى إلى 30 ميزة. على سبيل المثال، الحقل 3 هو نصف القطر المتوسط، والحقل 13 هو نصف القطر SE، والحقل 23 هو نصف القطر الأسوأ.
يتم إعادة ترميز جميع قيم الميزات بأربعة أرقام مهمة.  
قيم السمات المفقودة: لا شيء  
التوزيع الطبقي: 357 حميد، 212 خبيث  



## 2. تحليل البيانات الأولية



### 2.0 المعالجة المسبقة للبيانات


In [ ]:
target = pd.DataFrame(df['diagnosis'])
data = df.drop(['diagnosis'], axis=1)


### 2.1 الأعمدة الثابتة



نظرة عامة على البيانات:


In [ ]:
data.info()


قم بإسقاط العمود الثابت ** لم يذكر اسمه: 32 ** وعمود **المعرف** الذي لا فائدة منه للتحليل:


In [ ]:
data.drop(['Unnamed: 32', 'id'], axis=1, inplace=True)


### 2.2 القيم المفقودة



التحقق من البيانات بحثًا عن القيم المفقودة:


In [ ]:
print("Are there missing values:", data.isnull().values.any())


### 2.3 ملخص الإحصائيات



نظرة عامة على إحصاءات البيانات العامة:


In [ ]:
data.describe()


**الاستنتاج:** هنا يمكننا أن نرى قيمًا مختلفة للحد الأدنى/الحد الأقصى للميزات، على سبيل المثال *area_mean* و*smoothness_mean*. وبالتالي يجب علينا التحقق من القيم المتطرفة (مخطط الصندوق هو خيار جيد لذلك).



### 2.4 إحصائيات لفئات مختلفة



تحقق مما إذا كان اختلاف الميزات يعني أن القيم مهمة إحصائيًا. سوف نستخدم معايير مان ويتني، لأنها غير مناسبة للقيم المتطرفة وتوزيع العينات المختلفة.

In [ ]:
for column in data.columns:
    m = data[column][target['diagnosis']=='M']
    b = data[column][target['diagnosis']=='B']
    statistic, pvalue = mannwhitneyu(m, b)
    
    print('Column:', column, 'Important:', pvalue < 0.05 )


**الاستنتاج:** تعتبر الاختلافات في جميع الميزات تقريبًا ذات أهمية إحصائية. لذلك سوف يساهمون بالمزيد من المعلومات الكافية للتصنيف.



### 2.5 ميزة الهدف



عدد الايمبل لكل فئة:


In [ ]:
target['diagnosis'].value_counts()


دعونا نتحقق من نسبة الأمثلة التي تنتمي إلى كل فئة:


In [ ]:
target['diagnosis'].value_counts() / target['diagnosis'].size


**الخلاصة:** هناك الكثير من الأمثلة على الطبقة الحميدة، ولكنها ليست كافية لمشكلة الطبقات المنحرفة.



## 3. تحليل البيانات المرئية الأولية 



من أجل تصور البيانات الإعلامية، نحتاج إلى توحيد الميزات وتوسيع نطاقها، نظرًا لأن بعض الميزات لها قيم قصوى/دقيقة مختلفة جدًا.


In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
data_scaled = pd.DataFrame(scaled_data, columns=data.columns)
data_scaled['diagnosis'] = target['diagnosis']


### 3.1 التبعيات الخطية للميزات (مصفوفة الارتباط):



وظيفة مساعدة لتخطيط ارتباطات الميزات:


In [ ]:
def plot_corr(data):
    plt.figure(figsize=[40, 40])
    ax = sns.heatmap(data.corr(), annot=True, fmt= '.1f', linewidths=.5)
    ax.set_xticklabels(ax.get_xticklabels(), size='xx-large')
    ax.set_yticklabels(ax.get_yticklabels(), size='xx-large')
    plt.show();


ارتباطات البيانات:


In [ ]:
plot_corr(data)


**الاستنتاج:** هناك عدة مجموعات من الميزات المترابطة:
- نصف القطر، متوسط_المحيط، متوسط_المساحة 
- متوسط الاكتناز، متوسط التقعر، متوسط النقاط المقعرة
- radius_se، perimeter_se، Area_se
- نصف القطر_الأسوأ، والمحيط_الأسوأ، والمساحة_الأسوأ 
- الاكتناز_الأسوأ، التقعر_الأسوأ، النقاط المقعرة_الأسوأ
- Compactness_se، concavity_se، نقاط مقعرة_se
- الملمس_المتوسط، الملمس_الأسوأ
- منطقة_أسوأ، منطقة_متوسط



### 3.2 القيم المتطرفة


In [ ]:
data_z = pd.melt(data_scaled, id_vars="diagnosis", var_name="features", value_name='value')

In [ ]:
plt.figure(figsize=(20, 10));
ax = sns.boxplot(x='features', y='value', hue='diagnosis', data=data_z);
ax.set_xticklabels(ax.get_xticklabels());
plt.xticks(rotation=90);


**الاستنتاج:** هناك الكثير من المتغيرات ذات القيم المتطرفة. لذلك قبل التدريب علينا التعامل معها. 



### 3.3 توزيع الفصول


In [ ]:
plt.figure(figsize=(30, 20));
ax = sns.violinplot(x="features", y="value", hue="diagnosis", data=data_z, split=True, inner="quartile");
ax.set_xticklabels(ax.get_xticklabels(), size='large');
plt.xticks(rotation=90);


**الخلاصة:** في بعض الميزات، مثل *نصف القطر_mean*، *texture_mean*، متوسط كل فئة منفصلة، لذلك يمكن أن تكون مفيدة للتصنيف. الميزات الأخرى، مثل *smoothness_se*، ليست منفصلة تمامًا وستكون أقل فائدة للتصنيف. تتمتع معظم الميزات بتوزيع طبيعي وذيل طويل.


### 3.4 تقليل الأبعاد



تطبيق PCA لتقليل الأبعاد:


In [ ]:
pca = PCA(random_state=24)
pca.fit(scaled_data)

plt.figure(figsize=(10, 10))
plt.plot(pca.explained_variance_ratio_, linewidth=2)
plt.xlabel('Number of components');
plt.ylabel('Explained variance ratio');


**الخلاصة:** حسب طريقة الكوع يمكن اختيار 3 مكونات.



التحقق من عدد المكونات لشرح تباين البيانات:


In [ ]:
components = range(1, pca.n_components_ + 1)
plt.figure(figsize=(15, 5));
plt.bar(components, np.cumsum(pca.explained_variance_ratio_));
plt.hlines(y = .95, xmin=0, xmax=len(components), colors='green');


**الاستنتاج:** المكونان الأولان يفسران 0.6324 من التباين. نحتاج إلى 10 مكونات رئيسية لتفسير أكثر من 0.95 من التباين و17 لتفسير أكثر من 0.99. 



تقليل أبعاد البيانات ورسمها:


In [ ]:
pca_two_comp = PCA(n_components=2, random_state=24)
two_comp_data = pca_two_comp.fit_transform(scaled_data)
plt.scatter(x=two_comp_data[:, 0], y=two_comp_data[:, 1], 
            c=target['diagnosis'].map({'M': 'red', 'B': 'green'}))
plt.show()


**الاستنتاج:** البيانات جيدة بدرجة كافية ويمكن فصلها باستخدام مكونين فقط.



## 4. الرؤى والتبعيات الموجودة



ملخص البيانات:
- هناك الكثير من المجموعات ذات الميزات المترابطة. بعد ذلك يتعين علينا التخلص من العلاقة الخطية المتعددة عن طريق تحديد ميزة واحدة لكل مجموعة.
- نسبة الأمثلة في كل فئة 0.67/0.27. لا توجد فئات منحرفة هنا، وهو أمر مهم لاختيار المقياس؛
- الفروق في الخصائص الإحصائية (المتوسطة) لكل فئة ذات أهمية إحصائية. لذلك ستكون هذه الميزات مهمة للتصنيف.
- هناك قيم متطرفة في البيانات. من المهم التخلص منها بالنسبة للنماذج الحساسة للقيم المتطرفة (الانحدار اللوجستي على سبيل المثال) قبل التدريب؛
- يُظهر PCA أن البيانات جيدة بدرجة كافية ويمكن فصلها باستخدام 3-5 ميزات فقط.



## 5. اختيار المقاييس



التنبؤ بما إذا كان السرطان حميدًا أم خبيثًا هي مهمة **تصنيف ثنائي**. نحن هنا لا نواجه مشكلة الطبقات المنحرفة. لذلك سيكون مقياس **الدقة** خيارًا جيدًا لتقييم النموذج. كما أن هذا المقياس بسيط بما فيه الكفاية، وبالتالي قابل للتفسير بدرجة كبيرة.



$$Accuracy=\frac{Number~of~corrected~predictions}{Total~number~of~predictions}$$



أيضًا بالنسبة لمجموعة الاختبار، سنقوم بحساب **الدقة** و**الاستدعاء**.



## 6. اختيار النموذج


تم اختيار النموذج **الانحدار اللوجستي** للأسباب التالية:
- يعمل بشكل جيد مع الميزات غير الفئوية (في بياناتنا جميع الميزات مستمرة)؛
- قوية للضوضاء الصغيرة في البيانات؛
- يمكن التعامل مع حالات العلاقات الخطية المتداخلة المتعددة من خلال تنفيذ التنظيم؛
- يعمل بشكل جيد إذا لم تكن هناك بيانات مفقودة؛
- توافر التنفيذ الفعال؛
- مساحة الميزة للمهمة الحالية ليست كبيرة.



## 7. المعالجة المسبقة للبيانات



### 7.1 إسقاط الأعمدة عديمة الفائدة



قم بإسقاط العمود الثابت **غير مسمى: 32** والعمود عديم الفائدة **المعرف** للتصنيف.


In [ ]:
X = df.drop(['id', 'Unnamed: 32', 'diagnosis'], axis=1)
y = df['diagnosis'].map(lambda x: 1 if x=='M' else 0)


### 7.3 تقسيم البيانات إلى تدريب/اختبار



قم بتقسيم البيانات إلى تدريب/اختبار بنسبة 0.7/0.3 وهو تقسيم شائع لمثل هذه الكمية من البيانات.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=24)
print('Train size:', X_train.size)
print('Test size:', X_test.size)


### 7.2 اختيار الميزة



بادئ ذي بدء، ينبغي لنا أن نتعامل مع العلاقات الخطية المتداخلة المتعددة. من كل مجموعة من الميزات المترابطة سنختار ميزة واحدة فقط. إذن هنا الأعمدة التي يجب إسقاطها:


In [ ]:
corr_columns = ['perimeter_mean','radius_mean','compactness_mean',
                'concave points_mean','radius_se','perimeter_se',
                'radius_worst','perimeter_worst','compactness_worst',
                'concave points_worst','compactness_se','concave points_se',
                'texture_worst','area_worst',
                'concavity_mean']


قم بإسقاط الأعمدة المرتبطة من بيانات القطار:


In [ ]:
X_train = X_train.drop(corr_columns, axis=1)


قم بإسقاط الأعمدة المرتبطة من بيانات الاختبار:


In [ ]:
X_test = X_test.drop(corr_columns, axis=1)


التحقق من عدد الميزات المتبقية:


In [ ]:
print('Current number of features:', X_train.shape[1])


### 7.3 تحجيم الميزات


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 8. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج



استخدم 3 تقسيمات لأنه ليس لدينا كمية كبيرة من بيانات التدريب وقم بخلط العينات بترتيب عشوائي.


In [ ]:
cv = StratifiedKFold(n_splits=3, random_state=24)


نموذج:


In [ ]:
model = LogisticRegression(random_state=24)


معلمات النموذج:


In [ ]:
model_parameters = {'penalty': ['l1', 'l2'],
                    'C': np.linspace(.1, 1, 10)}


للعثور على أفضل المعلمات الفائقة، سنستخدم البحث الشبكي لأنه بسيط جدًا وفعال بدرجة كافية.


In [ ]:
grig_search = GridSearchCV(model, model_parameters, n_jobs=-1, cv=cv, scoring='accuracy')

In [ ]:
%%time
grig_search.fit(X_train_scaled, y_train);


أفضل معلمات النموذج:


In [ ]:
grig_search.best_params_


أفضل نتيجة السيرة الذاتية:


In [ ]:
print('Accuracy:', grig_search.best_score_)


## 9. إنشاء ميزات جديدة



وظيفة مساعدة لتطبيق عملية الخريطة على سمات إطار البيانات:


In [ ]:
def apply_cat_op(data, attrs, operation, prefix):
    """
    Apply one operation to data attributes.
    """
    series = [data[attr].map(operation) for attr in attrs]
    
    _data = pd.concat(series, axis=1).add_prefix(prefix)
    new_attrs = _data.columns.values
    
    return _data, new_attrs

يتطلب إنشاء ميزات جديدة تعتمد على الطب معرفة قوية بالمجال. لذلك سوف نقوم بإنشائها بناءً على الطبيعة الرياضية للميزات الحالية. يتمثل النهج الأساسي للميزات العددية لنموذج الانحدار في حساب مربعات الميزات من أجل التقاط التبعيات غير الخطية.



وظيفة مربعة:


In [ ]:
sq_operation = lambda x: x**2


قم بإنشاء ميزة مربعة لكل عمود واختبرها باستخدام النموذج:


In [ ]:
for column in X_train.columns:
    X_train_sq, sq_attr = apply_cat_op(X_train, [column], sq_operation, 'sq_')
    data = pd.concat([X_train, X_train_sq], axis=1)
    
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    grig_search = GridSearchCV(model, model_parameters, n_jobs=-1, cv=cv, scoring='accuracy')
    
    grig_search.fit(data_scaled, y_train);
    
    print('Column:', column, ' ', 
          'Accuracy:', grig_search.best_score_, ' ',
          'Best params:', grig_search.best_params_)


كما نرى ميزة التربيع *fractal_dimension_mean*، تعطي نتيجة أفضل باستخدام المعلمات {'C': 0.2, 'penalty': 'l2'}



إضافة ميزة جديدة لتدريب البيانات:


In [ ]:
X_train_sq, atr = apply_cat_op(X_train, ['fractal_dimension_mean'], sq_operation, 'sq_')
X_train = pd.concat([X_train, X_train_sq], axis=1)


إضافة ميزة جديدة لاختبار البيانات:


In [ ]:
X_test_sq, atr = apply_cat_op(X_test, ['fractal_dimension_mean'], sq_operation, 'sq_')
X_test = pd.concat([X_test, X_test_sq], axis=1)


#### قياس البيانات النهائية:


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#### نموذج القطار مع أفضل المعلمات في جميع بيانات القطار:


In [ ]:
final_model = LogisticRegression(penalty='l2', C=0.2)
final_model.fit(X_train_scaled, y_train)


## 10. رسم منحنيات التدريب والتحقق من الصحة



### 10.1 منحنى التدريب



رسم [وظيفة منحنى التعلم](https://scikit-learn.org/stable/auto_examples/model_selection/plot_learning_curve.html#sphx-glr-auto-examples-model-selection-plot-learning-curve-py):


In [ ]:
def plot_learning_curve(estimator, title, X, y, ylim=None, cv=None,
                        n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    """
    Generate a simple plot of the test and training learning curve.

    Parameters
    ----------
    estimator : object type that implements the "fit" and "predict" methods
        An object of that type which is cloned for each validation.

    title : string
        Title for the chart.

    X : array-like, shape (n_samples, n_features)
        Training vector, where n_samples is the number of samples and
        n_features is the number of features.

    y : array-like, shape (n_samples) or (n_samples, n_features), optional
        Target relative to X for classification or regression;
        None for unsupervised learning.

    ylim : tuple, shape (ymin, ymax), optional
        Defines minimum and maximum yvalues plotted.

    cv : int, cross-validation generator or an iterable, optional
        Determines the cross-validation splitting strategy.
        Possible inputs for cv are:
          - None, to use the default 3-fold cross-validation,
          - integer, to specify the number of folds.
          - :term:`CV splitter`,
          - An iterable yielding (train, test) splits as arrays of indices.

        For integer/None inputs, if ``y`` is binary or multiclass,
        :class:`StratifiedKFold` used. If the estimator is not a classifier
        or if ``y`` is neither binary nor multiclass, :class:`KFold` is used.

        Refer :ref:`User Guide <cross_validation>` for the various
        cross-validators that can be used here.

    n_jobs : int or None, optional (default=None)
        Number of jobs to run in parallel.
        ``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.
        ``-1`` means using all processors. See :term:`Glossary <n_jobs>`
        for more details.

    train_sizes : array-like, shape (n_ticks,), dtype float or int
        Relative or absolute numbers of training examples that will be used to
        generate the learning curve. If the dtype is float, it is regarded as a
        fraction of the maximum size of the training set (that is determined
        by the selected validation method), i.e. it has to be within (0, 1].
        Otherwise it is interpreted as absolute sizes of the training sets.
        Note that for classification the number of samples usually have to
        be big enough to contain at least one sample from each class.
        (default: np.linspace(0.1, 1.0, 5))
    """
    plt.figure()
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes)
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
             label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
             label="Cross-validation score")

    plt.legend(loc="best")
    return plt

In [ ]:
plot_learning_curve(final_model, 'Logistic regression', 
                    X_train_scaled, y_train, cv=cv);


**الاستنتاج:** تشير هذه الفجوة بين التدريب ومنحنى التحقق من الصحة إلى فرط التجهيز. ولكن يمكننا أن نرى أن منحنى التحقق يتزايد مع زيادة كمية أمثلة التدريب، لذلك من المرجح أن تساعد المزيد من البيانات في التغلب على التجاوز.



### 10.2 منحنى التحقق



رسم وظيفة منحنى التحقق من الصحة:


In [ ]:
def plot_validation_curve(estimator, title, X, y, param_name, param_range, 
                          cv=None, scoring=None, ylim=None, n_jobs=None):
    """
    Generates a simple plot of training and validation scores for different parameter values.
    
    Parameters
    ----------
    estimator : object type that implements the "fit" and "predict" methods
        An object of that type which is cloned for each validation.

    title : string
        Title for the chart.

    X : array-like, shape (n_samples, n_features)
        Training vector, where n_samples is the number of samples and
        n_features is the number of features.

    y : array-like, shape (n_samples) or (n_samples, n_features), optional
        Target relative to X for classification or regression;
        None for unsupervised learning.
    
    param_name : string
        Name of the parameter that will be varied.

    param_range : array-like, shape (n_values,)
        The values of the parameter that will be evaluated.

    cv : int, cross-validation generator or an iterable, optional
        Determines the cross-validation splitting strategy.
        Possible inputs for cv are:
          - None, to use the default 3-fold cross-validation,
          - integer, to specify the number of folds.
          - :term:`CV splitter`,
          - An iterable yielding (train, test) splits as arrays of indices.

        For integer/None inputs, if ``y`` is binary or multiclass,
        :class:`StratifiedKFold` used. If the estimator is not a classifier
        or if ``y`` is neither binary nor multiclass, :class:`KFold` is used.

        Refer :ref:`User Guide <cross_validation>` for the various
        cross-validators that can be used here.
    
    scoring : string, callable or None, optional, default: None
        A string (see model evaluation documentation) or
        a scorer callable object / function with signature
        ``scorer(estimator, X, y)``.
    
    ylim : tuple, shape (ymin, ymax), optional
        Defines minimum and maximum yvalues plotted.

    n_jobs : int or None, optional (default=None)
        Number of jobs to run in parallel.
        ``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.
        ``-1`` means using all processors. See :term:`Glossary <n_jobs>`
        for more details.
    
    """
    train_scores, test_scores = validation_curve(
    estimator, X, y, param_name, param_range,
    cv=cv, scoring=scoring, n_jobs=n_jobs)
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    plt.figure()
    plt.grid()
    plt.title(title)
    plt.xlabel(param_name)
    plt.ylabel("Score")
    if ylim is not None:
        plt.ylim(*ylim)
    plt.semilogx(param_range, train_scores_mean, 'o-', label="Training score",
                 color="darkorange")
    plt.fill_between(param_range, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.2,
                     color="darkorange")
    plt.semilogx(param_range, test_scores_mean, 'o-', label="Cross-validation score",
                 color="navy")
    plt.fill_between(param_range, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.2,
                     color="navy")
    plt.legend(loc="best")
    return plt


منحنى التحقق من صحة المؤامرة لمعلمة تعقيد النموذج:


In [ ]:
plot_validation_curve(final_model, 'Logistic regression', X_train_scaled, y_train,
                             'C', model_parameters['C'], 
                              cv=cv, scoring='accuracy');


**الاستنتاج:** الفجوة بين التدريب ومنحنى التحقق تشير إلى التجهيز الزائد. أفضل معلمة **C** هي 0.2



## 11. التنبؤ بعينات الاختبار



عمل تنبؤات لعينات الاختبار:


In [ ]:
test_predictions = final_model.predict(X_test_scaled)


#### درجة الدقة:


In [ ]:
print('Accuracy test score:', accuracy_score(y_test, test_predictions))


**الاستنتاج:** نتائج عينات الاختبار قابلة للمقارنة بنتائج التحقق المتبادل، بل إنها أفضل. وبالتالي فإن نظام التحقق الخاص بنا صالح.



#### مصفوفة الارتباك:


In [ ]:
test_confusion_matrix = confusion_matrix(test_predictions, y_test);
sns.heatmap(test_confusion_matrix, annot=True, fmt='d');

من مصفوفة الارتباك يمكننا أن نرى أننا قمنا ببعض التنبؤات الخاطئة.



#### الدقة:


In [ ]:
print('Precision:', precision_score(y_test, test_predictions))


#### أذكر:


In [ ]:
print('Recall:', recall_score(y_test, test_predictions))


## 12. الاستنتاجات



على الرغم من أننا نجرب نموذجًا بسيطًا، إلا أنه يوفر دقة بنسبة 98%، ودقة 98%، واستدعاء 97% لمجموعة الاختبار. هناك العديد من الميزات (3-5) الأكثر أهمية للتصنيف، والتي قد تشير إلى أن بياناتنا غير قابلة للتمثيل أو متحيزة. لذا، يعد تجربة النموذج بناءً على المزيد من البيانات خيارًا جيدًا. يعد إنشاء الميزات بناءً على المعرفة الطبية لمثل هذه البيانات أمرًا صعبًا للغاية، لذلك نقوم ببنائها بناءً على طبيعة الرياضيات. 



#### طرق التحسين:
- جمع المزيد من البيانات وإعادة تدريب النموذج عليها، حيث يمكننا أن نرى تحسنًا في درجة التحقق مع زيادة كمية البيانات على منحنى التعلم؛
- البحث في المجال وإنشاء المزيد من الميزات بناءً على الطب؛
- جرب نماذج أخرى، مثل الشبكة العصبية (للتقاط التبعيات غير الخطية المعقدة) أو الغابة العشوائية (القوية للتركيب الزائد)؛
- تطبيق PCA لتقليل أبعاد البيانات وتدريب النموذج على البيانات المخفضة؛
- حاول تكديس نماذج مختلفة.